In [32]:
import pandas as pd
from konlpy.tag import Mecab
from gensim import corpora
from gensim.models.ldamodel import LdaModel
import networkx as nx
import numpy as np
import tqdm

In [33]:
# --- 1단계: 데이터 준비 및 전처리 ---
print("1. 데이터 준비 및 전처리 시작...")


1. 데이터 준비 및 전처리 시작...


In [34]:
KINDS_PATH = '../data/interim/news/deep_search_news.csv'
STOPWORD_PATH = '../data/raw/news/stopwords-ko.txt'

POSITIVE_PATH = '../data/raw/news/sentiment/positive.txt'
NEGATIVE_PATH = '../data/raw/news/sentiment/negative.txt'
NATURAL_PATH = '../data/raw/news/sentiment/natural.txt'
# 뉴스 데이터 읽기
df = pd.read_csv(KINDS_PATH)
#df = data.head(20) # 테스트용으로 상위 20개만 해봄

# 각종 TXT 파일 불러오기
def load_txt(PATH):
    with open(PATH, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f]
# 불용어 불러오기
stopwords = load_txt(STOPWORD_PATH)
# 긍정, 부정, 중립 단어 불러오기
positive = load_txt(POSITIVE_PATH)
negative = load_txt(NEGATIVE_PATH)
natural = load_txt(NATURAL_PATH)

print(f"불용어 단어 예시 : {stopwords[:5]}...")
print(f"긍정 단어 예시 : {positive[:5]}...")
print(f"부정 단어 예시 : {negative[:5]}...")
print(f"중립 단어 예시 : {natural[:5]}...")

불용어 단어 예시 : ['가', '가까스로', '가령', '각', '각각']...
긍정 단어 예시 : ['활황', '급매물', '소진', '강세', '매수세']...
부정 단어 예시 : ['침체', '급매', '투매', '하락', '폭락']...
중립 단어 예시 : ['부동산', '아파트', '주택', '토지', '건물']...


In [35]:
df['content']

0        금융위원회는 이날 문자 공지를 통해 "은 위원장이 보유한 세종시 아파트에 대한 매수...
1        은성수 금융위원장이 8일 세종시에서 보유한 아파트의 매매 합의를 함에 따라 2주택자...
2        주택대출 규제의 주무 장관인 은성수 금융위원장이 8일 세종시에 보유한 아파트를 매도...
3        8일 금융권에 따르면 은 위원장은 8일 세종시 아파트 매매를 합의하고 가계약금을 수...
4        은 위원장은 지난해 12·16 부동산 대책 발표 후 ‘고위공직자 1주택 보유’ 기조...
                               ...                        
26739    서학개미 열풍 속에 올해 상반기 주요 증권사가 해외주식 거래 수수료로 벌어들인 수익...
26740    부동산원, 전국 주간 아파트 가격 동향 6·27 부동산 대책 발표 이후 서울의 아파...
26741    올해 3월 140.3㎡ 55억원에 매도 앞서 2017년 24.4억에 분양받아 축구선...
26742    보통주 총 1307만 5691주에 매각 가격은 주당 10만 7100원으로 총 1조 ...
26743    ‘청주 센텀 푸르지오 자이’ 8일(월) 무순위 청약 접수 약 1.4만여 가구 규모 ...
Name: content, Length: 26744, dtype: object

In [36]:
# 명사 추출 + 불용어 제거 함수
mecab = Mecab()
def tokenize(text):
    if not isinstance(text, str):
        return []
    return [word for word in mecab.nouns(text) 
            if len(word) > 1 and word not in stopwords]

# 토큰화 + 불용어 제거 적용
df['tokens'] = df['content'].apply(tokenize)

print(f"토큰 예시 : {df['tokens'][0]}")

토큰 예시 : ['금융', '위원회', '이날', '문자', '공지', '위원장', '보유', '종시', '아파트', '매수', '매매', '합의', '계약금', '수령', '당초', '위원장', '잠원동', '아파트', '도담동', '아파트', '본인', '명의', '종시', '아파트', '정세균', '주택', '보유', '권고', '처분', '호가', '수준', '매각', '위원장', '종시', '아파트', '최초', '매도', '호가', '수준', '계약', '성사']


In [37]:
## --- 2단계: 토픽 모델링 및 텍스트랭크 ---
print("\n2. 토픽 모델링 및 텍스트랭크 시작...")


2. 토픽 모델링 및 텍스트랭크 시작...


In [38]:
# 토픽 모델링 (LDA)
dictionary = corpora.Dictionary(df['tokens'])
corpus = [dictionary.doc2bow(tokens) for tokens in df['tokens']]
lda_model = LdaModel(corpus, num_topics=8, id2word=dictionary, passes=15)

topics = lda_model.print_topics(num_words=30)
print(f"토픽 모델링 결과 예시 :\n {topics[0]}")

토픽 모델링 결과 예시 :
 (0, '0.042*"사업" + 0.030*"재건축" + 0.029*"구역" + 0.024*"정비" + 0.022*"서울시" + 0.020*"아파트" + 0.018*"개발" + 0.017*"토지" + 0.017*"지정" + 0.016*"조합" + 0.016*"추진" + 0.016*"건축" + 0.016*"계획" + 0.012*"허가" + 0.012*"해제" + 0.011*"거래" + 0.011*"시공사" + 0.010*"주민" + 0.010*"지역" + 0.009*"선정" + 0.009*"공사비" + 0.009*"지구" + 0.007*"위원회" + 0.007*"입찰" + 0.006*"취소" + 0.006*"목동" + 0.006*"통과" + 0.006*"변경" + 0.005*"압구정" + 0.005*"조합원"')


In [39]:
# 텍스트랭크
def text_rank_keywords(tokens):
    g = nx.Graph()
    for i in range(len(tokens) - 1):
        g.add_edge(tokens[i], tokens[i+1])
    pr = nx.pagerank(g, weight='weight')
    return sorted(pr, key=pr.get, reverse=True) # 상위 몇개를 포함할건가?는 논문에 없다

In [40]:
df['textrank_keywords'] = df['tokens'].apply(text_rank_keywords)
print("\n텍스트랭크 키워드:")
print(df[['content', 'textrank_keywords']].head())


텍스트랭크 키워드:
                                             content  \
0  금융위원회는 이날 문자 공지를 통해 "은 위원장이 보유한 세종시 아파트에 대한 매수...   
1  은성수 금융위원장이 8일 세종시에서 보유한 아파트의 매매 합의를 함에 따라 2주택자...   
2  주택대출 규제의 주무 장관인 은성수 금융위원장이 8일 세종시에 보유한 아파트를 매도...   
3  8일 금융권에 따르면 은 위원장은 8일 세종시 아파트 매매를 합의하고 가계약금을 수...   
4  은 위원장은 지난해 12·16 부동산 대책 발표 후 ‘고위공직자 1주택 보유’ 기조...   

                                   textrank_keywords  
0  [아파트, 위원장, 보유, 종시, 수준, 호가, 위원회, 이날, 계약, 문자, 합의...  
1  [위원장, 아파트, 금융, 종시, 매매, 합의, 공지, 저녁, 문자, 오늘, 최근,...  
2  [아파트, 위원장, 종시, 주택, 매매, 금융, 부동산, 보유, 주무, 장관, 규제...  
3      [위원장, 아파트, 종시, 합의, 계약금, 매매, 수령, 지난해, 잠원동, 금융]  
4  [아파트, 위원장, 종시, 고위, 공직자, 발표, 주택, 대책, 보유, 부동산, 전...  


In [41]:
# --- 3단계: 감성 사전 기반 감성 점수 산출 ---
print("\n3. 감성 사전 기반 감성 점수 산출 시작...")

def get_sentiment_score(tokens):
    pos_score = sum(1 for word in tokens if word in positive)
    neg_score = sum(1 for word in tokens if word in negative)
    nat_score = sum(1 for word in tokens if word in natural)
    total_words = len(tokens)
    if total_words == 0:
        return 0
    return (pos_score - neg_score) / total_words

df['sentiment_score'] = df['tokens'].apply(get_sentiment_score)

print("\n감성 사전 기반 감성 점수:")
print(df[['content', 'sentiment_score']].head())


3. 감성 사전 기반 감성 점수 산출 시작...

감성 사전 기반 감성 점수:
                                             content  sentiment_score
0  금융위원회는 이날 문자 공지를 통해 "은 위원장이 보유한 세종시 아파트에 대한 매수...        -0.024390
1  은성수 금융위원장이 8일 세종시에서 보유한 아파트의 매매 합의를 함에 따라 2주택자...         0.030303
2  주택대출 규제의 주무 장관인 은성수 금융위원장이 8일 세종시에 보유한 아파트를 매도...        -0.093023
3  8일 금융권에 따르면 은 위원장은 8일 세종시 아파트 매매를 합의하고 가계약금을 수...         0.058824
4  은 위원장은 지난해 12·16 부동산 대책 발표 후 ‘고위공직자 1주택 보유’ 기조...        -0.037037


In [42]:
# --- 4단계: 월별 감성 지수 산출 및 예측 모델 통합 ---
print("\n4. 월별 감성 지수 산출 및 예측 모델 통합...")


4. 월별 감성 지수 산출 및 예측 모델 통합...


In [43]:
# datetime 변환
df['date'] = pd.to_datetime(df['date'], errors='coerce')  # 변환 불가 값은 NaT 처리
# 월 단위 추출
df['month'] = df['date'].dt.to_period('M')

In [44]:
monthly_sentiment = df.groupby('month')['sentiment_score'].mean().reset_index()
monthly_sentiment['month'] = monthly_sentiment['month'].astype(str)

# monthly_sentiment month 컬럼도 period[M]로 변환
monthly_sentiment['month'] = pd.to_datetime(monthly_sentiment['month']).dt.to_period('M')


In [45]:
print("\n월별 감성 지수:")
print(monthly_sentiment)
monthly_sentiment


월별 감성 지수:
      month  sentiment_score
0   2020-07         0.014421
1   2020-08         0.008881
2   2020-09         0.020459
3   2020-10         0.025977
4   2020-11         0.017741
..      ...              ...
58  2025-05         0.033125
59  2025-06         0.030115
60  2025-07         0.009352
61  2025-08         0.025129
62  2025-09         0.026038

[63 rows x 2 columns]


,month,sentiment_score
0,2020-07,0.014421
1,2020-08,0.008881
2,2020-09,0.020459
3,2020-10,0.025977
4,2020-11,0.017741
...,...,...
58,2025-05,0.033125
59,2025-06,0.030115
60,2025-07,0.009352
61,2025-08,0.025129


In [48]:
SALE_PATH = '../data/interim/apt/gang_nam_apt_with_long_lat.csv'
sale = pd.read_csv(SALE_PATH)

In [49]:
sale.head(1)

,단지명,전용면적(㎡),거래금액(만원),층,건축년도,도로명,면적당 단가(만원),아파트 나이,구,계약일자,계약년월,alpha,경도,위도
0,세곡리엔파크(3단지),84.96,115000,6,2011,헌릉로590길 11,7.210507,9,강남구,2020-07-11,202007,1.0,127.104003,37.464399


In [50]:
# datetime 변환
sale['계약일자'] = pd.to_datetime(sale['계약일자'], errors='coerce')  # 변환 불가 값은 NaT 처리
# 월 단위 추출
sale['month'] = sale['계약일자'].dt.to_period('M')
drop_col = ['단지명','도로명','경도','위도']

In [51]:
sale.columns

Index(['단지명', '전용면적(㎡)', '거래금액(만원)', '층', '건축년도', '도로명', '면적당 단가(만원)',
       '아파트 나이', '구', '계약일자', '계약년월', 'alpha', '경도', '위도', 'month'],
      dtype='object')

In [52]:
sale.drop(drop_col, axis=1, inplace=True)

In [53]:
sale.head()

,전용면적(㎡),거래금액(만원),층,건축년도,면적당 단가(만원),아파트 나이,구,계약일자,계약년월,alpha,month
0,84.96,115000,6,2011,7.210507,9,강남구,2020-07-11,202007,1.000000,2020-07
1,59.40,134000,4,1997,7.721301,23,강남구,2020-07-11,202007,0.233333,2020-07
2,84.83,139000,6,2013,7.401580,7,강남구,2020-07-11,202007,1.000000,2020-07
3,96.98,210000,12,1984,7.680358,36,강남구,2020-07-11,202007,0.000000,2020-07
4,111.96,91000,2,2004,6.700473,16,강남구,2020-07-11,202007,0.466667,2020-07


In [54]:
merged_df = pd.merge(sale, monthly_sentiment, on='month', how='left')

In [55]:
merged_df.drop('month', axis=1, inplace=True)

In [56]:
merged_df.head()

,전용면적(㎡),거래금액(만원),층,건축년도,면적당 단가(만원),아파트 나이,구,계약일자,계약년월,alpha,sentiment_score
0,84.96,115000,6,2011,7.210507,9,강남구,2020-07-11,202007,1.000000,0.014421
1,59.40,134000,4,1997,7.721301,23,강남구,2020-07-11,202007,0.233333,0.014421
2,84.83,139000,6,2013,7.401580,7,강남구,2020-07-11,202007,1.000000,0.014421
3,96.98,210000,12,1984,7.680358,36,강남구,2020-07-11,202007,0.000000,0.014421
4,111.96,91000,2,2004,6.700473,16,강남구,2020-07-11,202007,0.466667,0.014421


In [57]:
len(merged_df)

8907

In [58]:
merged_df.to_csv('../data/interim/gang_nam_sendimental_score_with_sale.csv',index=False)

In [24]:
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
import numpy as np


In [25]:

# 특성과 타깃 분리
X = merged_df[['전용면적(㎡)','층','건축년도','아파트 나이','alpha','sentiment_score']]
y = merged_df['면적당 단가(만원)']

# MLP 회귀 모델 + 표준화
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('mlp', MLPRegressor(hidden_layer_sizes=(32,16,8),
                         activation='relu',
                         solver='adam',
                         max_iter=500,
                         random_state=42))
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 10-fold 교차검증 (MAE)
scores = cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_absolute_error')

# MAE는 음수로 반환되므로 양수로 변환
mae_scores = -scores

print("10-fold CV MAE scores:", mae_scores)
print("Mean MAE:", np.mean(mae_scores))

10-fold CV MAE scores: [0.32901233 0.33292732 0.33243191 0.32902923 0.32926716 0.34074096
 0.33889897 0.33455075 0.33512781 0.33627461]
Mean MAE: 0.3338261047789882


In [26]:
sale

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,alpha,month
0,59.91,10,1998,6.766156,22,0.266667,2020-07
1,59.77,7,1996,7.499383,24,1.000000,2020-07
2,84.83,6,2013,7.401580,7,1.000000,2020-07
3,59.75,13,2016,7.066081,4,1.000000,2020-07
4,49.94,7,1989,6.967225,31,0.000000,2020-07
...,...,...,...,...,...,...,...
23568,54.34,2,1995,5.908227,30,0.000000,2025-06
23569,97.21,8,2006,7.406056,19,0.366667,2025-06
23570,23.70,11,2019,7.034407,6,0.800000,2025-06
23571,22.20,10,2002,7.139868,23,0.233333,2025-06


In [27]:

# 특성과 타깃 분리
X = sale[['전용면적(㎡)','층','건축년도','아파트 나이','alpha']]
y = sale['면적당 단가(만원)']

# MLP 회귀 모델 + 표준화
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('mlp', MLPRegressor(hidden_layer_sizes=(32,16,8),
                         activation='relu',
                         solver='adam',
                         max_iter=500,
                         random_state=42))
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 10-fold 교차검증 (MAE)
scores = cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_absolute_error')

# MAE는 음수로 반환되므로 양수로 변환
mae_scores = -scores

print("10-fold CV MAE scores:", mae_scores)
print("Mean MAE:", np.mean(mae_scores))

10-fold CV MAE scores: [0.3337105  0.33283429 0.3316904  0.33631928 0.33709114 0.33878993
 0.34406594 0.33747762 0.34000014 0.33981638]
Mean MAE: 0.3371795616117778


In [31]:
merged_df['sentiment_score'].to_csv('../data/interim/sendimental_score.csv', index=False)

In [32]:
len(merged_df)

23573